# Iceberg Schema Evolution Scenarios

This notebook demonstrates Apache Iceberg schema evolution patterns on AWS Glue:
1. Creating a base table
2. Adding columns
3. Dropping columns
4. Inserting old-schema data into the evolved table
5. Using branching for safe schema evolution
6. Loading old data that contains dropped columns

## Session Setup

Configure the Glue interactive session with Iceberg support.

In [21]:
%stop_session
# Update these values for your environment
%iam_role arn:aws:iam::038676220235:role/AWSGlueServiceNotebookRoleDefault

%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 300
%region us-west-2
%%configure
{
  "--datalake-formats": "iceberg",
  "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://<YOUR_BUCKET>/warehouse/ --conf spark.sql.catalog.glue_catalog.type=glue --conf spark.sql.defaultCatalog=glue_catalog"
}

Stopping session: 314b4b6f-928c-4b9c-a9f2-444b7f207147
Stopped session.
Current iam_role is arn:aws:iam::038676220235:role/AWSGlueServiceNotebookRoleDefault
iam_role has been set to arn:aws:iam::038676220235:role/AWSGlueServiceNotebookRoleDefault.
Setting Glue version to: 5.0
Previous worker type: G.1X
Setting new worker type to: G.1X
Previous number of workers: 2
Setting new number of workers to: 2
Current idle_timeout is 300 minutes.
idle_timeout has been set to 300 minutes.
Previous region: us-west-2
Setting new region to: us-west-2
Region is set to: us-west-2
The following configurations have been updated: {'--datalake-formats': 'iceberg', '--conf': 'spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://<YOUR_BUCKET>/warehouse/ --conf spark.sql.catalog.glue_catalog.type=glue --conf spark.sql.defaultCatalog=glue_catalog'}


In [ ]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col, current_timestamp
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType, IntegerType
from awsglue.context import GlueContext
from awsglue.job import Job

spark = SparkSession.builder.getOrCreate()
glueContext = GlueContext(spark.sparkContext)
job = Job(glueContext)

## Configuration

Set your database and table names here. Adjust the bucket/warehouse path in the session config above.

In [ ]:

DATABASE = "jpmc_demo_warehouse"
TABLE_NAME = "events"
FULL_TABLE = f"glue_catalog.{DATABASE}.{TABLE_NAME}"

# Create the database if it doesn't exist
#spark.sql(f"CREATE DATABASE IF NOT EXISTS glue_catalog.{DATABASE}")
print(f"Using database: {DATABASE}")

---
## Create Base Table

Start with a simple schema (v0)

In [81]:
# Drop table if exists (for re-runnability)
spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE}")

# Create the base Iceberg table
spark.sql(f"""
    CREATE TABLE {FULL_TABLE} (
        id BIGINT,
        name STRING,
        created_at TIMESTAMP
    )
    USING iceberg
""")

print(f"Created table: {FULL_TABLE}")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

Created table: glue_catalog.jpmc_demo_warehouse.events
+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|id        |bigint   |NULL   |
|name      |string   |NULL   |
|created_at|timestamp|NULL   |
+----------+---------+-------+


In [83]:
# Insert initial data (schema v0)
spark.sql(f"""
    INSERT INTO {FULL_TABLE} VALUES
        (1, 'alice', timestamp '2024-01-01 10:00:00'),
        (2, 'bob', timestamp '2024-01-02 11:00:00'),
        (3, 'charlie', timestamp '2024-01-03 12:00:00')
""")

print("Inserted 3 rows at schema v0")
spark.sql(f"SELECT * FROM {FULL_TABLE}").show()

Inserted 3 rows at schema v0
+---+-------+-------------------+
| id|   name|         created_at|
+---+-------+-------------------+
|  1|  alice|2024-01-01 10:00:00|
|  2|    bob|2024-01-02 11:00:00|
|  3|charlie|2024-01-03 12:00:00|
+---+-------+-------------------+


###### Add New Columns (Schema v0 -> v1)

Add  and  columns. Existing data will return  for these new columns.

In [85]:
# Add columns
spark.sql(f"""
    ALTER TABLE {FULL_TABLE} ADD COLUMNS (
        region STRING,
        event_type STRING
    )
""")

print("Schema after adding columns:")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

Schema after adding columns:
+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|id        |bigint   |NULL   |
|name      |string   |NULL   |
|created_at|timestamp|NULL   |
|region    |string   |NULL   |
|event_type|string   |NULL   |
+----------+---------+-------+


In [87]:
# Verify: old rows show null for new columns
print("Old rows now show null for new columns:")
spark.sql(f"SELECT * FROM {FULL_TABLE}").show()

Old rows now show null for new columns:
+---+-------+-------------------+------+----------+
| id|   name|         created_at|region|event_type|
+---+-------+-------------------+------+----------+
|  1|  alice|2024-01-01 10:00:00|  NULL|      NULL|
|  2|    bob|2024-01-02 11:00:00|  NULL|      NULL|
|  3|charlie|2024-01-03 12:00:00|  NULL|      NULL|
+---+-------+-------------------+------+----------+


In [89]:
# Insert new data with all columns (schema v1)
spark.sql(f"""
    INSERT INTO {FULL_TABLE} VALUES
        (4, 'diana', timestamp '2024-02-01 09:00:00', 'us-east-1', 'login'),
        (5, 'eve', timestamp '2024-02-02 10:00:00', 'eu-west-1', 'purchase')
""")

print("Inserted 2 rows at schema v1 (with region and event_type)")
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show()

Inserted 2 rows at schema v1 (with region and event_type)
+---+-------+-------------------+---------+----------+
| id|   name|         created_at|   region|event_type|
+---+-------+-------------------+---------+----------+
|  1|  alice|2024-01-01 10:00:00|     NULL|      NULL|
|  2|    bob|2024-01-02 11:00:00|     NULL|      NULL|
|  3|charlie|2024-01-03 12:00:00|     NULL|      NULL|
|  4|  diana|2024-02-01 09:00:00|us-east-1|     login|
|  5|    eve|2024-02-02 10:00:00|eu-west-1|  purchase|
+---+-------+-------------------+---------+----------+


In [91]:
spark.sql(f"""
SELECT * from {FULL_TABLE}.snapshots """).show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                    |summary                                                           

---
## Using Branching for Safe Schema Evolution

Create a branch before major schema changes, then write OLD schema data to branch, validate



In [93]:
# Create a branch from the current state
BRANCH_NAME = "branch_schema_test"
spark.sql(f"""
    ALTER TABLE {FULL_TABLE}
    CREATE BRANCH {BRANCH_NAME}

""")

print(f"Created branch '{BRANCH_NAME}'")

Created branch 'branch_schema_test'


In [95]:
# Drop event_type column --> table schema on main
spark.sql(f"ALTER TABLE {FULL_TABLE} DROP COLUMN event_type")

print("Schema after dropping event_type:")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

Schema after dropping event_type:
+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|id        |bigint   |NULL   |
|name      |string   |NULL   |
|created_at|timestamp|NULL   |
|region    |string   |NULL   |
+----------+---------+-------+


##### event_type is gone from query results (data files still have it physically):

In [97]:
# Verify: event_type no longer appears in reads from main branch

spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show(truncate=False)

+---+-------+-------------------+---------+
|id |name   |created_at         |region   |
+---+-------+-------------------+---------+
|1  |alice  |2024-01-01 10:00:00|NULL     |
|2  |bob    |2024-01-02 11:00:00|NULL     |
|3  |charlie|2024-01-03 12:00:00|NULL     |
|4  |diana  |2024-02-01 09:00:00|us-east-1|
|5  |eve    |2024-02-02 10:00:00|eu-west-1|
+---+-------+-------------------+---------+


##### event_type is AVAILABLE from query results from test_branch

In [121]:
spark.sql(f"SELECT * FROM {FULL_TABLE}.refs  ").show(truncate=False)

+------------------+------+-------------------+-----------------------+---------------------+----------------------+
|name              |type  |snapshot_id        |max_reference_age_in_ms|min_snapshots_to_keep|max_snapshot_age_in_ms|
+------------------+------+-------------------+-----------------------+---------------------+----------------------+
|branch_schema_test|BRANCH|501710803880282164 |NULL                   |NULL                 |NULL                  |
|main              |BRANCH|4664838866992857609|NULL                   |NULL                 |NULL                  |
+------------------+------+-------------------+-----------------------+---------------------+----------------------+


In [125]:
# Verify: event_type exists on test branch --> The query can only be performed using snapshot references

spark.sql(f"SELECT * FROM {FULL_TABLE} VERSION AS OF 4664838866992857609 ORDER BY id").show(truncate=False)

+---+-------+-------------------+---------+----------+
|id |name   |created_at         |region   |event_type|
+---+-------+-------------------+---------+----------+
|1  |alice  |2024-01-01 10:00:00|NULL     |NULL      |
|2  |bob    |2024-01-02 11:00:00|NULL     |NULL      |
|3  |charlie|2024-01-03 12:00:00|NULL     |NULL      |
|4  |diana  |2024-02-01 09:00:00|us-east-1|login     |
|5  |eve    |2024-02-02 10:00:00|eu-west-1|purchase  |
+---+-------+-------------------+---------+----------+


---
## Insert Old-Schema Data into the Evolved Table
WARNING: Iceberg cannot write to a branch with different schema than the main branch. All writes need to conform to the current schema.

In [103]:
# enable WAP ()
spark.sql(f"""ALTER TABLE {FULL_TABLE} SET TBLPROPERTIES (
    'write.wap.enabled'='true');""")

DataFrame[]


Simulate loading data that only has the original schema  --> This fails, cannot write using old schema


In [105]:
# Write OLD data to the branch only
spark.conf.set("spark.wap.branch", BRANCH_NAME)

branch_data = spark.createDataFrame([
    (500, "branch_user_1", "2025-04-01 10:00:00", "us-east-1", 'buy'),
    (501, "branch_user_2", "2025-04-02 11:00:00", "eu-west-1", 'trade'),
], ["id", "name", "created_at", "region", "event_type"])

branch_data = branch_data.withColumn("id", col("id").cast(LongType())).withColumn("created_at", col("created_at").cast(TimestampType()))

branch_data.writeTo(FULL_TABLE).append()

# Reset WAP branch
spark.conf.unset("spark.wap.branch")

print(f"Wrote 2 rows to branch '{BRANCH_NAME}'")

AnalysisException: [INSERT_COLUMN_ARITY_MISMATCH.TOO_MANY_DATA_COLUMNS] Cannot write to `glue_catalog`.`jpmc_demo_warehouse`.`events`, the reason is too many data columns:
Table columns: `id`, `name`, `created_at`, `region`.
Data columns: `id`, `name`, `created_at`, `region`, `event_type`.


Simulate loading data that only has the original schema, but drop invalid columns


In [107]:
# Write OLD data to the branch only --> USE NEW MAIN SCHEMA,
# in other words, old columns no longer valid need to be removed
spark.conf.set("spark.wap.branch", BRANCH_NAME)

branch_data = spark.createDataFrame([
    (500, "branch_user_3", "2025-04-01 10:00:00", "us-east-1", 'buy'),
    (501, "branch_user_4", "2025-04-02 11:00:00", "eu-west-1", 'trade'),
], ["id", "name", "created_at", "region", "event_type"])

branch_data = branch_data.withColumn("id", col("id").cast(LongType())).withColumn("created_at", col("created_at").cast(TimestampType()))

branch_data = branch_data.drop("event_type")

branch_data.writeTo(FULL_TABLE).append()

# Reset WAP branch
spark.conf.unset("spark.wap.branch")

print(f"Wrote 2 rows to branch '{BRANCH_NAME}'")

Wrote 2 rows to branch 'branch_schema_test'


In [129]:
spark.sql(f"SELECT * FROM {FULL_TABLE}.branch_{BRANCH_NAME} ORDER BY id").show(truncate=False)

+---+-------------+-------------------+---------+
|id |name         |created_at         |region   |
+---+-------------+-------------------+---------+
|1  |alice        |2024-01-01 10:00:00|NULL     |
|2  |bob          |2024-01-02 11:00:00|NULL     |
|3  |charlie      |2024-01-03 12:00:00|NULL     |
|4  |diana        |2024-02-01 09:00:00|us-east-1|
|5  |eve          |2024-02-02 10:00:00|eu-west-1|
|500|branch_user_3|2025-04-01 10:00:00|us-east-1|
|501|branch_user_4|2025-04-02 11:00:00|eu-west-1|
+---+-------------+-------------------+---------+


In [127]:
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show(truncate=False)

+---+-------+-------------------+---------+
|id |name   |created_at         |region   |
+---+-------+-------------------+---------+
|1  |alice  |2024-01-01 10:00:00|NULL     |
|2  |bob    |2024-01-02 11:00:00|NULL     |
|3  |charlie|2024-01-03 12:00:00|NULL     |
|4  |diana  |2024-02-01 09:00:00|us-east-1|
|5  |eve    |2024-02-02 10:00:00|eu-west-1|
+---+-------+-------------------+---------+


---
## Scenario 7: Loading Old Data That Contains Dropped Columns

The table has dropped  (back in Scenario 3). Now we need to load data that still has that column.

Iceberg rejects writes for columns not in the current schema, so we must handle the mismatch.

### Approach A: Drop extra columns from the data before insert (lossy)

In [ ]:
# Simulate old data that has the dropped column event_type
old_data_with_dropped_cols = spark.createDataFrame([
    (600, "old_full_1", "2023-01-10 08:00:00", "us-west-2", "click", 5),
    (601, "old_full_2", "2023-01-11 09:00:00", "eu-west-1", "view", 2),
], ["id", "name", "created_at", "region", "event_type", "priority"])

old_data_with_dropped_cols = old_data_with_dropped_cols     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("priority", col("priority").cast(IntegerType()))

# Get current table columns and select only those
current_columns = [f.name for f in spark.table(FULL_TABLE).schema.fields]
print(f"Current table columns: {current_columns}")

# Drop columns not in current schema
trimmed_df = old_data_with_dropped_cols.select(current_columns)
print("Trimmed DataFrame (event_type removed):")
trimmed_df.show()

trimmed_df.writeTo(FULL_TABLE).append()
print("Inserted (event_type discarded):")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 600 ORDER BY id").show()

### Approach B: Re-add dropped columns temporarily, load, then drop again (lossless)

In [ ]:
# Re-add the dropped column temporarily
spark.sql(f"ALTER TABLE {FULL_TABLE} ADD COLUMNS (event_type STRING)")

print("Schema after re-adding event_type:")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

In [ ]:
# Now we can insert data that includes event_type
old_data_full_schema = spark.createDataFrame([
    (700, "archive_full_1", "2022-08-01 07:00:00", "ap-south-1", 3, "signup"),
    (701, "archive_full_2", "2022-08-02 08:00:00", "us-east-1", 1, "logout"),
], ["id", "name", "created_at", "region", "priority", "event_type"])

old_data_full_schema = old_data_full_schema     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("priority", col("priority").cast(IntegerType()))

old_data_full_schema.writeTo(FULL_TABLE).append()

print("Inserted with event_type preserved:")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 700 ORDER BY id").show()

In [ ]:
# Drop event_type again to restore the production schema
spark.sql(f"ALTER TABLE {FULL_TABLE} DROP COLUMN event_type")

print("Schema restored (event_type dropped again):")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

print("Data still readable (event_type hidden):")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 700 ORDER BY id").show()

### Approach C: Use a staging table

In [ ]:
STAGING_TABLE = f"glue_catalog.{DATABASE}.events_staging"

# Create staging table with the old full schema
spark.sql(f"DROP TABLE IF EXISTS {STAGING_TABLE}")
spark.sql(f"""
    CREATE TABLE {STAGING_TABLE} (
        id BIGINT,
        name STRING,
        created_at TIMESTAMP,
        region STRING,
        event_type STRING,
        priority INT
    )
    USING iceberg
""")

print(f"Created staging table: {STAGING_TABLE}")

In [ ]:
# Load old data into staging (preserves all columns including dropped ones)
staging_data = spark.createDataFrame([
    (800, "staged_1", "2022-05-01 06:00:00", "us-west-1", "download", 4),
    (801, "staged_2", "2022-05-02 07:00:00", "eu-central-1", "upload", 2),
], ["id", "name", "created_at", "region", "event_type", "priority"])

staging_data = staging_data     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("priority", col("priority").cast(IntegerType()))

staging_data.writeTo(STAGING_TABLE).append()

print("Staging table has full old-schema data:")
spark.sql(f"SELECT * FROM {STAGING_TABLE}").show()

In [ ]:
# Insert into production selecting only current columns
current_columns_str = ", ".join([f.name for f in spark.table(FULL_TABLE).schema.fields])
print(f"Inserting columns: {current_columns_str}")

spark.sql(f"""
    INSERT INTO {FULL_TABLE} ({current_columns_str})
    SELECT {current_columns_str} FROM {STAGING_TABLE}
""")

print("Production table after staging insert:")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 800 ORDER BY id").show()

### Approach D: Branch + Temporary Schema Expansion

Combine branching with temporary column re-addition for zero-downtime backfill.

In [ ]:
# 1. Create a branch for the backfill
spark.sql(f"""
    ALTER TABLE {FULL_TABLE}
    CREATE BRANCH backfill_old_data
    RETAIN 7 DAYS
""")

# 2. Re-add dropped column
spark.sql(f"ALTER TABLE {FULL_TABLE} ADD COLUMNS (event_type STRING)")

print("Branch created, event_type re-added for backfill")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

In [ ]:
# 3. Write old data to the branch
spark.conf.set("spark.wap.branch", "backfill_old_data")

backfill_data = spark.createDataFrame([
    (900, "branched_backfill_1", "2022-01-15 05:00:00", "us-east-2", 5, "api_call"),
    (901, "branched_backfill_2", "2022-01-16 06:00:00", "ap-northeast-1", 3, "webhook"),
], ["id", "name", "created_at", "region", "priority", "event_type"])

backfill_data = backfill_data     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("priority", col("priority").cast(IntegerType()))

backfill_data.writeTo(FULL_TABLE).append()
spark.conf.unset("spark.wap.branch")

print("Wrote backfill data to branch")

In [ ]:
# 4. Drop event_type again (hides from future reads)
spark.sql(f"ALTER TABLE {FULL_TABLE} DROP COLUMN event_type")

# 5. Validate on the branch
print("Branch data (event_type hidden after drop):")
spark.read.option("branch", "backfill_old_data").table(FULL_TABLE)     .filter("id >= 900").show()

print("Main still unchanged:")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 900").show()

In [ ]:
# 6. Fast-forward main
spark.sql(f"""
    CALL glue_catalog.system.fast_forward(
        table => '{DATABASE}.{TABLE_NAME}',
        branch => 'main',
        to => 'backfill_old_data'
    )
""")

print("Fast-forwarded main. Backfill data now visible:")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 900 ORDER BY id").show()

In [ ]:
# 7. Clean up branch
spark.sql(f"ALTER TABLE {FULL_TABLE} DROP BRANCH backfill_old_data")
print("Dropped branch 'backfill_old_data'")

---
## Final State & Summary

In [113]:
# Show final table state
print("=== Final Schema ===")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

print(f"=== Total Row Count ===")
spark.sql(f"SELECT count(*) as total_rows FROM {FULL_TABLE}").show()

print("=== Sample of All Data ===")
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show(50, truncate=False)

=== Final Schema ===
+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|id        |bigint   |NULL   |
|name      |string   |NULL   |
|created_at|timestamp|NULL   |
|region    |string   |NULL   |
+----------+---------+-------+

=== Total Row Count ===
+----------+
|total_rows|
+----------+
|         5|
+----------+

=== Sample of All Data ===
+---+-------+-------------------+---------+
|id |name   |created_at         |region   |
+---+-------+-------------------+---------+
|1  |alice  |2024-01-01 10:00:00|NULL     |
|2  |bob    |2024-01-02 11:00:00|NULL     |
|3  |charlie|2024-01-03 12:00:00|NULL     |
|4  |diana  |2024-02-01 09:00:00|us-east-1|
|5  |eve    |2024-02-02 10:00:00|eu-west-1|
+---+-------+-------------------+---------+


In [115]:
# View schema evolution history via snapshots
print("=== Snapshot History ===")
spark.sql(f"SELECT * FROM {FULL_TABLE}.snapshots").show(truncate=False)

=== Snapshot History ===
+-----------------------+-------------------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                    |summary                                  

---
## Cleanup (Optional)

Uncomment and run to drop the demo tables.

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {STAGING_TABLE}")
# spark.sql(f"DROP DATABASE IF EXISTS glue_catalog.{DATABASE}")
# print("Cleaned up all demo resources")